In [5]:
import numpy as np
from scipy import signal
from sklearn.linear_model import LinearRegression

In [ ]:
n_time = 300          
n_nodes = 268         
n_motion = 12         

rng = np.random.default_rng(0)

# fake node time series: time x nodes
bold = rng.normal(size=(n_time, n_nodes))

# fake nuisance regressors: time x regressors
motion = rng.normal(size=(n_time, n_motion))
wm = rng.normal(size=(n_time, 1))
csf = rng.normal(size=(n_time, 1))
global_signal = bold.mean(axis=1, keepdims=True)

# linear drift term
linear_trend = np.linspace(-1, 1, n_time).reshape(-1, 1)

# combining nuisance regressors
confounds = np.hstack([
    motion,
    wm,
    csf,
    global_signal,
    linear_trend
])

In [7]:
model = LinearRegression()
model.fit(confounds, bold)
predicted_noise = model.predict(confounds)
bold_clean = bold - predicted_noise

In [ ]:
TR = 0.72                 
fs = 1 / TR               
cutoff = 0.1              

b, a = signal.butter(N=2, Wn=cutoff,btype="lowpass",fs=fs)

bold_filtered = signal.filtfilt(b,a,bold_clean,axis=0)

In [10]:
print(bold.shape)
print(motion.shape)
print(confounds.shape)

(300, 268)
(300, 12)
(300, 16)


In [ ]:
corr_matrix = np.corrcoef(bold_filtered, rowvar=False)

# avoid infinities before Fisher z
corr_matrix = np.clip(corr_matrix, -0.999999, 0.999999)

fisher_z_matrix = np.arctanh(corr_matrix)

# usually set diagonal to 0 because self-correlation is not useful
np.fill_diagonal(fisher_z_matrix, 0)